# Inspect Evo2 Top-20 Logits

This notebook loads a real GTDB sequence, runs `evo2_7b`, and prints the top-20 next-token logits/probabilities at selected positions.

Notes:
- The tokenizer is character-level, so `A/T/C/G/N` are ordinary character IDs.
- `logits[pos]` predicts the token at `pos + 1`, not the token at `pos` itself.
- Running this notebook requires GPU memory for `evo2_7b`.

In [2]:
from __future__ import annotations

import gzip
import json
import sys
from collections import Counter
from pathlib import Path

import torch
import torch.nn.functional as F
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None

ROOT = Path.cwd()
if ROOT.name != 'Llamascopium':
    raise RuntimeError(f'Please open this notebook from the repo root. Current cwd: {ROOT}')

SRC_ROOT = ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from llamascopium.backend.evo2 import DEFAULT_EVO2_CHECKPOINTS, load_evo2

MODEL_NAME = 'evo2_7b'
LOCAL_MODEL_PATH = DEFAULT_EVO2_CHECKPOINTS[MODEL_NAME]
RAW_JSONL_GZ = Path('/inspire/hdd/project/reasoning/public/activations/evo2_7b/opengenome2/json/midtraining_specific/gtdb_v220_stitched/data_gtdb_train_chunk1.jsonl.gz')
ROW_INDEX = 0
PROMPT_START = 0
PROMPT_LENGTH = 512
CUSTOM_PROMPT = None

# Negative positions are allowed and will be resolved relative to prompt length.
POSITION_SPECS = [0, 1, 10, 50, 100, 200, -3, -2, -1]
TOP_K = 20
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DNA_TOKENS = ['A', 'T', 'C', 'G', 'N']

print(f'MODEL_NAME={MODEL_NAME}')
print(f'LOCAL_MODEL_PATH={LOCAL_MODEL_PATH}')
print(f'RAW_JSONL_GZ={RAW_JSONL_GZ}')
print(f'DEVICE={DEVICE}')

MODEL_NAME=evo2_7b
RAW_JSONL_GZ=/inspire/hdd/project/reasoning/public/activations/evo2_7b/opengenome2/json/midtraining_specific/gtdb_v220_stitched/data_gtdb_train_chunk1.jsonl.gz
DEVICE=cuda:0


In [3]:
def load_prompt_from_jsonl(path: Path, row_index: int, start: int, length: int) -> str:
    with gzip.open(path, 'rt') as f:
        for i, line in enumerate(f):
            if i == row_index:
                row = json.loads(line)
                text = row['text']
                return text[start : start + length]
    raise IndexError(f'row_index={row_index} is out of range for {path}')


prompt = CUSTOM_PROMPT or load_prompt_from_jsonl(
    RAW_JSONL_GZ,
    row_index=ROW_INDEX,
    start=PROMPT_START,
    length=PROMPT_LENGTH,
)

assert len(prompt) >= 2, 'Prompt must have length >= 2 so logits[pos] has a next-token target.'

char_counts = Counter(prompt)
print(f'prompt_length={len(prompt)}')
print(f'prompt_preview={prompt[:120]}')
print('char_counts=', char_counts)
print('non_ATCGN_chars=', sorted(set(prompt) - set(DNA_TOKENS)))

prompt_length=512
prompt_preview=GCTACGACACGAACGATCTCGATACTATTCCGGCTGCATCAATCGAATCATTTGCCGGTGTGGGCTACTTCTTCGATCTCGCCGGTATAAAAGAAGGCGATAACATACTTGACTTAGGAA
char_counts= Counter({'A': 151, 'G': 124, 'T': 120, 'C': 117})
non_ATCGN_chars= []


In [4]:
if not LOCAL_MODEL_PATH.exists():
    raise FileNotFoundError(f'Local Evo2 checkpoint not found: {LOCAL_MODEL_PATH}')

evo2 = load_evo2(MODEL_NAME, local_path=LOCAL_MODEL_PATH)
tokenizer = evo2.tokenizer

input_ids = torch.tensor(tokenizer.tokenize(prompt), dtype=torch.long, device=DEVICE).unsqueeze(0)
with torch.no_grad():
    logits, _ = evo2(input_ids)

logits = logits[0].detach().float().cpu()
token_ids = input_ids[0].detach().cpu()
dna_token_ids = {token: tokenizer.tokenize(token)[0] for token in DNA_TOKENS}

print(f'input_ids.shape={tuple(input_ids.shape)}')
print(f'logits.shape={tuple(logits.shape)}')
print('dna_token_ids=', dna_token_ids)

LocalEntryNotFoundError: An error happened while trying to locate the files on the Hub and we cannot find the appropriate snapshot folder for the specified revision on the local disk. Please check your internet connection and try again.

In [ ]:
def resolve_positions(position_specs: list[int], prompt_length: int) -> list[int]:
    resolved = []
    max_valid = prompt_length - 1
    for spec in position_specs:
        pos = spec if spec >= 0 else prompt_length + spec
        if not (0 <= pos <= max_valid):
            raise ValueError(f'Invalid position spec {spec} -> {pos} for prompt_length={prompt_length}')
        resolved.append(pos)
    return resolved


def decode_token_id(token_id: int) -> str:
    if token_id == 0:
        return '<EOD>'
    if token_id == 1:
        return '<PAD>'
    if token_id == 9:
        return '\\t'
    if token_id == 10:
        return '\\n'
    if 32 <= token_id <= 126:
        return chr(token_id)
    return f'<{token_id}>'


def extract_topk_table(position: int, top_k: int = TOP_K):
    probs = F.softmax(logits[position], dim=-1)
    top_probs, top_token_ids = torch.topk(probs, k=top_k)
    top_logits = logits[position][top_token_ids]

    dna_mass = float(probs[list(dna_token_ids.values())].sum().item())
    other_mass = 1.0 - dna_mass
    target_id = token_ids[position + 1].item() if position + 1 < len(token_ids) else None

    rows = []
    for rank, (token_id, logit_value, prob_value) in enumerate(
        zip(top_token_ids.tolist(), top_logits.tolist(), top_probs.tolist()),
        start=1,
    ):
        rows.append(
            {
                'rank': rank,
                'token_id': token_id,
                'token': decode_token_id(token_id),
                'logit': float(logit_value),
                'prob': float(prob_value),
                'is_ATCGN': token_id in dna_token_ids.values(),
                'is_true_next': target_id is not None and token_id == target_id,
            }
        )

    return {
        'position': position,
        'current_token_id': token_ids[position].item(),
        'current_token': decode_token_id(token_ids[position].item()),
        'true_next_token_id': target_id,
        'true_next_token': decode_token_id(target_id) if target_id is not None else None,
        'top1_token_id': top_token_ids[0].item(),
        'top1_token': decode_token_id(top_token_ids[0].item()),
        'top1_prob': float(top_probs[0].item()),
        'dna_mass': dna_mass,
        'other_mass': other_mass,
        'rows': rows,
    }


def show_position(position: int, window: int = 24):
    result = extract_topk_table(position)
    left = max(0, position - window)
    right = min(len(prompt), position + window + 1)
    context = prompt[left:right]
    marker = ' ' * (position - left) + '^'

    print('=' * 100)
    print(f'position={result["position"]} current_token={result["current_token"]!r} true_next={result["true_next_token"]!r}')
    print(f'top1={result["top1_token"]!r} top1_prob={result["top1_prob"]:.6f}')
    print(f'p(ATCGN)={result["dna_mass"]:.6f}  p(other)={result["other_mass"]:.6f}')
    print(context)
    print(marker)

    if pd is not None:
        display(pd.DataFrame(result['rows']))
    else:
        for row in result['rows']:
            print(row)

    return result

In [ ]:
resolved_positions = resolve_positions(POSITION_SPECS, len(prompt))
print('resolved_positions=', resolved_positions)

results = []
for position in resolved_positions:
    results.append(show_position(position))